## Import necessary packages

In [17]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load Dataset

In [24]:
df = pd.read_csv("../dataset/rootnepalerp_smart_restock_training_dataset.csv")


In [25]:
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (11100, 39)


,date,interval_days,product_id,sku_number,product_name,category_id,category_name,supplier_id,supplier_name,supplier_company,...,quantity_purchased,current_stock,stockout_flag,avg_daily_sales_interval,days_until_stockout_est,sales_value,purchase_value,inventory_value,gross_margin_value,suggested_restock_qty
0,2024-02-14,5,12,SK-100,Himalayan Herbal Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,21,30,0,3.2,9.4,16000.0,7350.0,10500.00,10400.00,25
1,2024-02-14,5,13,SK-101,Tibetan Monastery Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,40,47,0,4.4,10.7,22000.0,14000.0,16450.00,14300.00,35
2,2024-02-14,5,14,SK-102,Tibetan Peace Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,30,30,1,2.8,10.7,11200.0,5550.0,5550.00,8610.00,26
3,2024-02-14,5,15,SK-103,Ribo Sangtsheo Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,0,9,0,1.8,5.0,7200.0,0.0,1664.82,5535.18,0
4,2024-02-14,5,16,SK-104,The Earth Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,40,46,0,2.2,20.9,8800.0,7400.0,8510.00,6765.00,31


## Optional Feature Engineering

In [26]:
df["profit_margin"] = df["selling_price"] - df["cost_price"]
df["stock_gap"] = df["reorder_level"] - df["current_stock"]

# Avoid division by zero
df["sales_to_stock_ratio"] = df["quantity_sold"] / df["current_stock"].replace(0, 1)

# Weekend flag from day_of_week if available
weekend_days = ["Saturday", "Sunday"]
df["is_weekend"] = df["day_of_week"].isin(weekend_days).astype(int)

df.head()

,date,interval_days,product_id,sku_number,product_name,category_id,category_name,supplier_id,supplier_name,supplier_company,...,days_until_stockout_est,sales_value,purchase_value,inventory_value,gross_margin_value,suggested_restock_qty,profit_margin,stock_gap,sales_to_stock_ratio,is_weekend
0,2024-02-14,5,12,SK-100,Himalayan Herbal Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,9.4,16000.0,7350.0,10500.00,10400.00,25,650.00,-25,0.533333,0
1,2024-02-14,5,13,SK-101,Tibetan Monastery Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,10.7,22000.0,14000.0,16450.00,14300.00,35,650.00,-42,0.468085,0
2,2024-02-14,5,14,SK-102,Tibetan Peace Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,10.7,11200.0,5550.0,5550.00,8610.00,26,615.00,-27,0.466667,0
3,2024-02-14,5,15,SK-103,Ribo Sangtsheo Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,5.0,7200.0,0.0,1664.82,5535.18,0,615.02,-6,1.000000,0
4,2024-02-14,5,16,SK-104,The Earth Incense,9,Tibetan Incense,4,Dipendra Dangol,Lovely Incense,...,20.9,8800.0,7400.0,8510.00,6765.00,31,615.00,-43,0.239130,0


## Choose Input Features (X) ans Target (Y)

In [27]:
# Target column
target_column = "suggested_restock_qty"

# Features to use
feature_columns = [
    "product_id",
    "category_name",
    "supplier_name",
    "selling_price",
    "cost_price",
    "current_stock",
    "reorder_level",
    "lead_time_days",
    "quantity_sold",
    "quantity_purchased",
    "stockout_flag",
    "day_of_week",
    "month",
    "season",
    "holiday_flag",
    "is_peak_tourist_season",
    "sales_value",
    "purchase_value",
    "inventory_value",
    "profit_margin",
    "stock_gap",
    "sales_to_stock_ratio",
    "is_weekend",
]

X = df[feature_columns]
y = df[target_column]

print("\nX shape:", X.shape)
print("y shape:", y.shape)


X shape: (11100, 23)
y shape: (11100,)


## Identify numeric and categorical columns

In [28]:
categorical_features = [
    "category_name",
    "supplier_name",
    "day_of_week",
    "season",
]

numeric_features = [
    "product_id",
    "selling_price",
    "cost_price",
    "current_stock",
    "reorder_level",
    "lead_time_days",
    "quantity_sold",
    "quantity_purchased",
    "stockout_flag",
    "month",
    "holiday_flag",
    "is_peak_tourist_season",
    "sales_value",
    "purchase_value",
    "inventory_value",
    "profit_margin",
    "stock_gap",
    "sales_to_stock_ratio",
    "is_weekend",
]

## Preprocessing

In [29]:
# Numeric: fill missing values with median
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical: fill missing values + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [30]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['product_id', 'selling_price', 'cost_price',
                                  'current_stock', 'reorder_level',
                                  'lead_time_days', 'quantity_sold',
                                  'quantity_purchased', 'stockout_flag',
                                  'month', 'holiday_flag',
                                  'is_peak_tourist_season', 'sales_value',
                                  'purchase_value', 'inventory_value',
                                  'profit_margin', 'stock_gap',
                                  'sales_to_stock_ratio', 'is_weekend']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['category_name', 'supplier_name',
                                  'day_of_week', 'season'])])

## Train/ Test Split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("\nTrain size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (8880, 23)
Test size: (2220, 23)


## Build Model Pipeline

In [32]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['product_id',
                                                   'selling_price',
                                                   'cost_price',
                                                   'current_stock',
                                                   'reorder_level',
                                                   'lead_time_days',
                                                   'quantity_sold',
                                                   'quantity_purchased',
                                                   'stockout_flag', 'month',
                                                   'holiday_flag',
                                                   'is_peak_tourist_season',
                                                   'sales_value',
                                                   'purchase...
                                                   'sales_to_stock_ratio',
                                                   'is_weekend']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category_name',
                                                   'supplier_name',
                                                   'day_of_week',
                                                   'season'])])),
                ('model',
                 RandomForestRegressor(max_depth=12, min_samples_leaf=2,
                                       min_samples_split=5, n_estimators=200,
                                       n_jobs=-1, random_state=42))])

## Train Model

In [33]:
pipeline.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


## Evaluate Model

In [34]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nEvaluation Results")
print("------------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))


Evaluation Results
------------------
MAE : 0.5
RMSE: 2.35
R²  : 0.9915


## Compare Actual VS Predicted

In [35]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": np.round(y_pred, 2)
})

print("\nSample Predictions:")
print(results.head(10))


Sample Predictions:
   Actual  Predicted
0       0       0.00
1       0       0.00
2       0       0.00
3       0       0.00
4      42      42.10
5      20      20.04
6       0       0.00
7      24      25.21
8       0       0.00
9       0       0.00


In [36]:
# joblib.dump(pipeline, "smart_restock_random_forest.pkl")
# print("\nModel saved as smart_restock_random_forest.pkl")

In [37]:
joblib.dump(pipeline, "../backend/ml/model/smart_restock_random_forest.pkl")

['../backend/ml/model/smart_restock_random_forest.pkl']